# Частина 3. Зовнішня перевірка запущеного API

Цей notebook — **незалежний зовнішній клієнт**. Він:

- надсилає справжні HTTP-запити до окремо запущеного сервера;
- **не імпортує** `app`, обробники запитів, правила перевірки чи сховище;
- **не використовує** `TestClient`;
- **не викликає** провайдера LLM напряму — ключ потрібен лише серверу.

Очікування беруться зі специфікації `specs/001-route-catalog/spec.md` та
зафіксованих випадків `evals/cases.json`, а не обчислюються функцією, яку
перевіряє.

**Навіщо це окремий notebook.** Тести сервісу перевіряють код на контрольованих
даних. Цей notebook додатково перевіряє те, чого тести не бачать: що
**окремо запущений** сервер доступний через HTTP, що HTTP-маршрути, передавання
JSON і спостережуваний результат у сховищі працюють так, як сказано у
специфікації.

---

## Порядок проходу

Перевірка потребує **трьох запусків сервера**, щоб кожен сценарій мав своє
джерело відповіді AI-компонента.

| Крок | Режим сервера | Сховище | Сценарії |
|---|---|---|---|
| 1 | `ROUTE_LLM_MODE=stub` | `data/test-routes.json` | список, ручне створення, хибний ручний вхід, усі AI-сценарії |
| 2 | `ROUTE_LLM_MODE=offline` | `data/test-routes-offline.json` | збій моделі й незалежний ручний шлях |
| 3 | `ROUTE_LLM_MODE=live` | `data/test-routes-live.json` | жива інтеграція через сервіс |

Команди запуску — у `README.md`, розділ «Перевірка через HTTP».

**Перед кожним повним проходом** видаляйте файл тестового сховища: `Restart
Kernel → Run All` скидає результати notebook, але **не очищає серверне
сховище**.

## 1. Умови запуску

- Python 3.11+ та стандартна бібліотека (жодних додаткових пакетів для HTTP).
- Сервер запущено окремо, на `http://127.0.0.1:8000`.
- Адреса й маршрути нижче відповідають специфікації.

In [1]:
import json
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

BASE_URL = "http://127.0.0.1:8000"
LIST_PATH = "/routes"
MANUAL_PATH = "/routes"
AI_PATH = "/routes/from-text"

# Таймаут має враховувати граничний час обробки AI-запиту, визначений у сервісі.
HTTP_TIMEOUT_SECONDS = 30

LIST_HTTP = 200
CREATED_HTTP = 201
REJECTED_HTTP = 422
MODEL_DOWN_HTTP = 503

CASE_RESULTS: list[dict] = []

## 2. HTTP-клієнт

`call_api(method, path, payload)` надсилає запит і повертає `(код, тіло)`.
`HTTPError` — це **відповідь сервера**, її треба розбирати, а не ховати:
невдала перевірка повинна бути видимою. `URLError` і таймаут означають, що
сервер недоступний — це інша категорія, ніж помилка відповіді.

In [2]:
def call_api(method: str, path: str, payload=None) -> tuple[int, object]:
    """Надсилає HTTP-запит. Повертає (код, тіло)."""
    data = None if payload is None else json.dumps(payload, ensure_ascii=False).encode("utf-8")
    request = Request(
        BASE_URL.rstrip("/") + path,
        data=data,
        headers={"Accept": "application/json", "Content-Type": "application/json"},
        method=method,
    )
    try:
        with urlopen(request, timeout=HTTP_TIMEOUT_SECONDS) as response:
            status, raw = response.status, response.read()
    except HTTPError as error:
        with error:
            status, raw = error.code, error.read()
    except (URLError, TimeoutError) as error:
        raise ConnectionError(f"Сервер недоступний: {error}") from error

    if not raw:
        return status, None
    try:
        return status, json.loads(raw)
    except json.JSONDecodeError:
        return status, {"_raw": raw.decode("utf-8", errors="replace")}


def get_records() -> list[dict]:
    """Список записів. Перевіряє код і структуру, повертає items."""
    status, body = call_api("GET", LIST_PATH)
    assert status == LIST_HTTP, f"очікувався HTTP {LIST_HTTP}, отримано {status}"
    assert isinstance(body, dict), f"очікувався об'єкт, отримано {type(body).__name__}"
    assert isinstance(body.get("items"), list), "у відповіді немає масиву items"
    assert body.get("count") == len(body["items"]), "count != len(items)"
    return body["items"]


def snapshot(items: list[dict]) -> list[tuple]:
    """Знімок стану сховища: ідентифікатори й поля.

    Для перевірки відмови цього недостатньо: кількість може не змінитися,
    поки запис було перезаписано. Тому порівнюємо поля.
    """
    return [
        (item["id"], item["name"], item["distance_km"], item["difficulty"], item["duration_hours"])
        for item in items
    ]

## 3. Помічник перевірок

`run_check(name, check)` виконує перевірку й записує результат у
`CASE_RESULTS`: `passed` або `failed` з причиною. Одна невдала перевірка не
зупиняє решту — інакше вже отримані спостереження були б приховані.

`assert` — це сама перевірка: надрукована відповідь без порівняння з
очікуванням перевіркою **не** є.

In [3]:
class NotRun(Exception):
    """Доказ відсутній: перевірку виконати не вдалося. Це не провал."""


def run_check(name: str, check) -> None:
    """Виконує перевірку і записує результат разом із фактами.

    Перевірка повертає словник із фактичними спостереженнями: кодом HTTP,
    зміною кількості записів та коротким підсумком. Ці факти потрібні для
    підсумкової таблиці: «перевірка виконана» — це твердження, а «код 201,
    записів +1» — це спостереження.
    """
    try:
        facts = check() or {}
    except NotRun as error:
        row = {"scenario": name, "status": "not_run", "reason": str(error), "http": "—", "delta": "—"}
    except AssertionError as error:
        row = {"scenario": name, "status": "failed", "reason": str(error), "http": "—", "delta": "—"}
    except Exception as error:  # noqa: BLE001
        row = {
            "scenario": name,
            "status": "failed",
            "reason": f"{type(error).__name__}: {error}",
            "http": "—",
            "delta": "—",
        }
    else:
        row = {
            "scenario": name,
            "status": "passed",
            "reason": "усі очікування виконано",
            "http": facts.get("http", "—"),
            "delta": facts.get("delta", "—"),
        }
    CASE_RESULTS.append(row)
    print(f"{row['status']:<7} {row['scenario']}: {row['reason']} | HTTP {row['http']} | записів {row['delta']}")


print("Сценаріїв виконано:", len(CASE_RESULTS))

Сценаріїв виконано: 0


## 4. Список і початковий стан

Перший запит — до будь-яких створень. Він фіксує початковий стан каталогу, з
яким порівнюватимуться всі наступні сценарії.

In [4]:
def check_initial_state() -> None:
    status, body = call_api("GET", LIST_PATH)
    assert status == LIST_HTTP, f"очікувався HTTP {LIST_HTTP}, отримано {status}"
    assert isinstance(body, dict), "відповідь має бути об'єктом"
    assert isinstance(body["items"], list), "items має бути масивом"
    assert body["count"] == len(body["items"]), "count має дорівнювати len(items)"

    # Порядок визначено специфікацією: за зростанням id.
    ids = [item["id"] for item in body["items"]]
    assert ids == sorted(ids), f"порядок порушено: {ids}"

    for item in body["items"]:
        assert set(item) == {
            "id", "name", "distance_km", "difficulty", "duration_hours", "created_at"
        }, f"неочікувані поля в записі: {sorted(item)}"
        assert isinstance(item["id"], int)
        assert 0.5 <= item["distance_km"] <= 100.0
        assert 0.5 <= item["duration_hours"] <= 24.0
        assert item["difficulty"] in ("easy", "medium", "hard")
        assert 3 <= len(item["name"]) <= 80

    return {"http": status, "delta": f"{len(ids)} (без змін)"}


run_check("Список і початковий стан", check_initial_state)

INITIAL_IDS = {item["id"] for item in get_records()}
print("Початковий стан зафіксовано, записів:", len(INITIAL_IDS))

passed  Список і початковий стан: усі очікування виконано | HTTP 200 | записів 0 (без змін)
Початковий стан зафіксовано, записів: 0


## 5. Ручне створення

Правильні поля, **без звернення до моделі**. Перевіряється не лише код, а й
те, що запис став видимим у списку, поля збігаються з введеними, а кількість
зросла рівно на один.

In [5]:
MANUAL_INPUT = {
    "name": "Ручний маршрут",
    "distance_km": 8.0,
    "difficulty": "easy",
    "duration_hours": 2.5,
}
EXPECTED_FIELDS = {
    "name": "Ручний маршрут",
    "distance_km": 8.0,
    "difficulty": "easy",
    "duration_hours": 2.5,
}


def check_manual_create() -> None:
    before = snapshot(get_records())

    status, created = call_api("POST", MANUAL_PATH, MANUAL_INPUT)
    assert status == CREATED_HTTP, f"очікувався HTTP {CREATED_HTTP}, отримано {status}"
    assert isinstance(created, dict), "відповідь має бути об'єктом"
    assert "id" in created, "у відповіді немає id"
    assert isinstance(created["id"], int), "id має бути цілим числом"
    for key, value in EXPECTED_FIELDS.items():
        assert created[key] == value, f"поле {key}: {created[key]!r} != {value!r}"
    assert "created_at" in created, "у відповіді немає created_at"

    after = snapshot(get_records())
    assert len(after) == len(before) + 1, (
        f"кількість записів: було {len(before)}, стало {len(after)}"
    )
    assert created["id"] not in {item[0] for item in before}, "новий id уже був у списку"
    assert created["id"] in {item[0] for item in after}, "нового записа немає у списку"

    # Поля в списку мають збігатися з відповіддю на створення.
    listed = [item for item in get_records() if item["id"] == created["id"]]
    assert len(listed) == 1, "новий запис має бути рівно один"
    for key, value in EXPECTED_FIELDS.items():
        assert listed[0][key] == value, f"у списку {key} != {value!r}"

    return {"http": status, "delta": f"{len(before)} -> {len(after)} (+1)"}


run_check("Ручне створення", check_manual_create)

passed  Ручне створення: усі очікування виконано | HTTP 201 | записів 0 -> 1 (+1)


## 6. Хибний ручний вхід

Поля, що порушують фіксоване правило (`distance_km` поза діапазоном
0.5–100.0). Очікується визначена відмова — HTTP `422` — і **незмінність
попередніх записів**.

Порівняння йде за ідентифікаторами й полями, а не лише за кількістю:
кількість записів не змінилася б і тоді, коли б запис було перезаписано.

In [6]:
INVALID_MANUAL_INPUT = {
    "name": "Хибний маршрут",
    "distance_km": 0.1,
    "difficulty": "easy",
    "duration_hours": 2.0,
}


def check_invalid_manual() -> None:
    before = snapshot(get_records())

    status, body = call_api("POST", MANUAL_PATH, INVALID_MANUAL_INPUT)
    assert status == REJECTED_HTTP, f"очікувався HTTP {REJECTED_HTTP}, отримано {status}"
    assert isinstance(body, dict) and body, "сервер має пояснити відмову"
    assert body.get("status") == "rejected", f"очікувався status=rejected, отримано {body.get('status')!r}"
    assert body.get("reason"), "у відповіді немає reason"
    assert body.get("detail"), "у відповіді немає detail"

    after = snapshot(get_records())
    assert after == before, "сховище не змінилося"

    return {"http": status, "delta": f"{len(before)} -> {len(after)} (без змін)"}


run_check("Хибний ручний вхід", check_invalid_manual)

passed  Хибний ручний вхід: усі очікування виконано | HTTP 422 | записів 1 -> 1 (без змін)


## 7. AI-сценарії

Тут сервер працює в режимі `stub`: замість живого виклику він повертає
підготовлену відповідь із `stubs/responses.json` за **точним** текстом
запиту. Підміна відбувається всередину сервера — цей notebook надсилає
звичайні HTTP-запити й не знає, як працює підміна.

Перевіряються чотири сценарії:

| Сценарій | Пропозиція моделі | Очікування |
|---|---|---|
| Коректна AI-пропозиція | усі поля правильні | `201`, рівно один новий запис, поля збігаються |
| Неповний або неоднозначний текст | немає `duration_hours` | `422`, `needs_clarification`, `missing` заповнений, сховище не змінене |
| Структурно хибна пропозиція | `distance_km` не число | `422`, `rejected`, `invalid_structure` |
| Змістовно хибна пропозиція | `distance_km = 21`, у тексті 18 | `422`, `rejected`, `invalid_semantics` |

In [7]:
TEXT_VALID = "Маршрут Соколиця: 12.5 км, середня складність, орієнтовно 4 години."
TEXT_INCOMPLETE = "Маршрут Дубовий Гай, 15 км, складність дуже висока."
TEXT_STRUCTURAL = "Маршрут Кринична Гора: 10 км, легкий, 2 години."
TEXT_SEMANTIC = "Маршрут Тиха Гавань, 18 км, середня складність, 5 годин."

AI_EXPECTED = {
    "name": "Соколиця",
    "distance_km": 12.5,
    "difficulty": "medium",
    "duration_hours": 4.0,
}


def check_ai_create() -> None:
    before = snapshot(get_records())

    status, created = call_api("POST", AI_PATH, {"text": TEXT_VALID})
    assert status == CREATED_HTTP, f"очікувався HTTP {CREATED_HTTP}, отримано {status}"
    assert isinstance(created, dict) and "id" in created, "відповідь має містити id"
    for key, value in AI_EXPECTED.items():
        assert created[key] == value, f"поле {key}: {created[key]!r} != {value!r}"

    after = snapshot(get_records())
    assert len(after) == len(before) + 1, (
        f"кількість записів: було {len(before)}, стало {len(after)}"
    )
    new_ids = {item[0] for item in after} - {item[0] for item in before}
    assert new_ids == {created["id"]}, f"нових записів: {new_ids}, очікувався {created['id']}"

    return {"http": status, "delta": f"{len(before)} -> {len(after)} (+1)"}


def check_ai_incomplete() -> None:
    before = snapshot(get_records())

    status, body = call_api("POST", AI_PATH, {"text": TEXT_INCOMPLETE})
    assert status == REJECTED_HTTP, f"очікувався HTTP {REJECTED_HTTP}, отримано {status}"
    assert body.get("status") == "needs_clarification", f"очікувався needs_clarification, отримано {body.get('status')!r}"
    assert body.get("missing") == ["duration_hours"], f"очікувався missing=['duration_hours'], отримано {body.get('missing')!r}"
    assert body.get("questions"), "клієнт має отримати питання до автора"
    assert "record" not in body, "у відповіді не має бути готового запису"

    after = snapshot(get_records())
    assert after == before, "сховище не змінилося"

    return {"http": status, "delta": f"{len(before)} -> {len(after)} (без змін)"}


def check_ai_structural() -> None:
    before = snapshot(get_records())

    status, body = call_api("POST", AI_PATH, {"text": TEXT_STRUCTURAL})
    assert status == REJECTED_HTTP, f"очікувався HTTP {REJECTED_HTTP}, отримано {status}"
    assert body.get("status") == "rejected", f"очікувався rejected, отримано {body.get('status')!r}"
    assert "invalid_structure" in body.get("reason", ""), f"причина: {body.get('reason')!r}"

    after = snapshot(get_records())
    assert after == before, "сховище не змінилося"

    return {"http": status, "delta": f"{len(before)} -> {len(after)} (без змін)"}


def check_ai_semantic() -> None:
    before = snapshot(get_records())

    status, body = call_api("POST", AI_PATH, {"text": TEXT_SEMANTIC})
    assert status == REJECTED_HTTP, f"очікувався HTTP {REJECTED_HTTP}, отримано {status}"
    assert body.get("status") == "rejected", f"очікувався rejected, отримано {body.get('status')!r}"
    assert "invalid_semantics" in body.get("reason", ""), f"причина: {body.get('reason')!r}"
    assert "distance_km" in body.get("reason", ""), "причина має вказувати поле"

    after = snapshot(get_records())
    assert after == before, "сховище не змінилося"

    return {"http": status, "delta": f"{len(before)} -> {len(after)} (без змін)"}


run_check("Коректна AI-пропозиція", check_ai_create)
run_check("Неповний або неоднозначний текст", check_ai_incomplete)
run_check("Структурно хибна пропозиція", check_ai_structural)
run_check("Змістовно хибна пропозиція", check_ai_semantic)

passed  Коректна AI-пропозиція: усі очікування виконано | HTTP 201 | записів 1 -> 2 (+1)
passed  Неповний або неоднозначний текст: усі очікування виконано | HTTP 422 | записів 2 -> 2 (без змін)
passed  Структурно хибна пропозиція: усі очікування виконано | HTTP 422 | записів 2 -> 2 (без змін)


passed  Змістовно хибна пропозиція: усі очікування виконано | HTTP 422 | записів 2 -> 2 (без змін)


## 8. Збій моделі й незалежний ручний шлях

**Крок 2 проходу.** Перезапустіть сервер у режимі `offline` з окремим
сховищем:

```powershell
$env:ROUTE_LLM_MODE = "offline"
$env:ROUTES_DB = "data/test-routes-offline.json"
.venv\Scripts\python.exe -m uvicorn app.main:app --port 8000
```

Очікується: визначений результат збою (`503`, `model_unavailable`) **без зміни
даних**, а ручний шлях — створення та список — **продовжує працювати** і
підтверджує новий ручний запис.

Якщо сервер ще працює в режимі `stub`, сценарій покаже `failed` — це очікувано
для цього кроку, бо підміна відповіді моделі не імітує збій.

In [8]:
def check_model_failure() -> None:
    before = snapshot(get_records())

    status, body = call_api("POST", AI_PATH, {"text": TEXT_VALID})
    ai_status = status
    assert status == MODEL_DOWN_HTTP, (
        f"очікувався HTTP {MODEL_DOWN_HTTP} (збій моделі), отримано {status}"
    )
    assert body.get("status") == "model_unavailable", (
        f"очікувався model_unavailable, отримано {body.get('status')!r}"
    )
    assert body.get("reason"), "у відповіді немає reason"

    after = snapshot(get_records())
    assert after == before, "сховище не змінилося"

    # Незалежний ручний шлях досі працює і підтверджує новий запис.
    status, created = call_api("POST", MANUAL_PATH, {
        "name": "Після збою моделі",
        "distance_km": 6.0,
        "difficulty": "medium",
        "duration_hours": 1.5,
    })
    assert status == CREATED_HTTP, f"ручний шлях зламався: HTTP {status}"
    items = get_records()
    assert created["id"] in {item["id"] for item in items}, "новий ручний запис не видно у списку"
    assert any(item["name"] == "Після збою моделі" for item in items), "новий ручний запис відсутній"

    # У фактах — код AI-запиту (503), а не ручного створення (201).
    return {
        "http": f"{ai_status} (AI), {status} (ручний)",
        "delta": f"{len(before)} -> {len(items)} (AI: без змін, ручний: +1)",
    }


run_check("Збій моделі й незалежний ручний шлях", check_model_failure)

passed  Збій моделі й незалежний ручний шлях: усі очікування виконано | HTTP 503 (AI), 201 (ручний) | записів 0 -> 1 (AI: без змін, ручний: +1)


## 9. Жива інтеграція через сервіс

**Крок 3 проходу.** Перезапустіть сервер у режимі `live` з окремим сховищем:

```powershell
$env:ROUTE_LLM_MODE = "live"
$env:ROUTES_DB = "data/test-routes-live.json"
.venv\Scripts\python.exe -m uvicorn app.main:app --port 8000
```

Цей сценарій показує **зв'язок сервісу з провайдером**. Він не доводить
стабільності моделі на інших входах, і підготовлена відповідь не замінює його.

Тут важлива деталь: сценарій використовує текст, якого **немає** у файлі
підготовлених відповідей (`stubs/responses.json`). Тому:

- у режимі `live` сервер іде до справжнього провайдера — перевірка має
  пройти;
- у режимі `stub` сервер не знає відповіді для цього тексту й повертає
  `invalid_structure`. Тоді сценарій позначається `not_run` з поясненням, що
  сервер запущено не в тому режимі, — а **не** `passed`. Інакше перевірка
  виглядала б як доказ живої інтеграції, хоча її не було.

Якщо провайдер недоступний, сценарій має бути позначений `not_run` — це
**відсутній доказ**, а не успіх.

In [9]:
def check_live_integration() -> None:
    # Цей текст навмисно НЕ є у stubs/responses.json: у режимі live сервер
    # іде до справжнього провайдера, а у режимі stub не знає відповіді.
    live_text = "Маршрут Жива Перевірка: 5.5 км, середня складність, 1.5 години."
    before = snapshot(get_records())

    status, body = call_api("POST", AI_PATH, {"text": live_text})
    if status == MODEL_DOWN_HTTP:
        raise NotRun(f"провайдер недоступний ({body.get('reason')})")
    if status == REJECTED_HTTP and "invalid_structure" in body.get("reason", ""):
        raise NotRun(
            "сервер працює не в режимі live: для цього тексту немає підготовленої відповіді"
        )
    assert status == CREATED_HTTP, f"очікувався HTTP {CREATED_HTTP}, отримано {status}: {body}"
    assert isinstance(body, dict) and "id" in body, "відповідь має містити id"
    assert body["distance_km"] == 5.5, f"distance_km: {body.get('distance_km')!r}"
    assert body["difficulty"] == "medium", f"difficulty: {body.get('difficulty')!r}"
    assert body["duration_hours"] == 1.5, f"duration_hours: {body.get('duration_hours')!r}"
    assert "Жива" in body["name"], f"name: {body.get('name')!r}"

    after = snapshot(get_records())
    assert len(after) == len(before) + 1, "живий виклик мав створити рівно один запис"

    return {"http": status, "delta": f"{len(before)} -> {len(after)} (+1)"}


try:
    run_check("Жива інтеграція через сервіс", check_live_integration)
except ConnectionError as error:
    CASE_RESULTS.append({
        "scenario": "Жива інтеграція через сервіс",
        "status": "not_run",
        "reason": f"сервер недоступний: {error}",
    })
    print(f"not_run  Жива інтеграція через сервіс: сервер недоступний: {error}")

passed  Жива інтеграція через сервіс: усі очікування виконано | HTTP 201 | записів 0 -> 1 (+1)


## 10. Підсумок і межа доказу

Таблиця нижче формується з фактичних перевірок. Сценарії, яких не було
виконано, позначаються `not_run`.

**Що доводить кожен результат:**

- `passed` — сервер повівся за специфікацією, перевірка виконала всі
  очікування, зміни записів відповідають очікуванню;
- `failed` — поведінка сервера не відповідає специфікації або доказ
  відсутній (сервер недоступний, неправильний режим підміни);
- `not_run` — доказ **відсутній**: перевірку виконати не вдалося або
  провайдер недоступний. Це не успіх.

In [10]:
REQUIRED_SCENARIOS = [
    ("Список і початковий стан", "AC2", "не застосовується",
     "HTTP 200; items — масив; count == len(items); порядок за id"),
    ("Ручне створення", "AC1", "не застосовується",
     "HTTP 201; поля відповіді == введені; новий id є в списку; записів +1"),
    ("Хибний ручний вхід", "AC3", "не застосовується",
     "HTTP 422; status=rejected; сховище не змінене за id і полями"),
    ("Коректна AI-пропозиція", "AC4", "підготовлена відповідь",
     "HTTP 201; name/distance_km/difficulty/duration_hours як очікувалися; записів +1"),
    ("Неповний або неоднозначний текст", "AC5", "підготовлена відповідь",
     "HTTP 422; status=needs_clarification; missing=['duration_hours']; запису немає"),
    ("Структурно хибна пропозиція", "AC6", "підготовлена відповідь",
     "HTTP 422; status=rejected; reason містить invalid_structure; запису немає"),
    ("Змістовно хибна пропозиція", "AC7", "підготовлена відповідь",
     "HTTP 422; status=rejected; reason містить invalid_semantics і distance_km; запису немає"),
    ("Збій моделі й незалежний ручний шлях", "AC8", "імітований збій",
     "HTTP 503; status=model_unavailable; сховище не змінене; ручний шлях створює запис"),
    ("Жива інтеграція через сервіс", "AC9", "реальна модель",
     "HTTP 201; поля як очікувалися; записів +1"),
]

latest = {row["scenario"]: row for row in CASE_RESULTS}

table = (
    "| Критерій | Сценарій | Джерело відповіді | Очікування | Фактичний HTTP | "
    "Зміни записів | Стан | Причина |\n"
    "|---|---|---|---|---|---|---|---|\n"
)
for name, criterion, source, expected in REQUIRED_SCENARIOS:
    row = latest.get(name)
    if row is None:
        table += f"| {criterion} | {name} | {source} | {expected} | — | — | not_run | сценарій не виконано |\n"
        continue
    reason = row["reason"].replace("|", "/").replace("\n", " ")
    table += (
        f"| {criterion} | {name} | {source} | {expected} | {row.get('http', '—')} | "
        f"{row.get('delta', '—')} | {row['status']} | {reason} |\n"
    )

print(table)

passed = sum(1 for row in CASE_RESULTS if row["status"] == "passed")
failed = sum(1 for row in CASE_RESULTS if row["status"] == "failed")
not_run = sum(1 for row in CASE_RESULTS if row["status"] == "not_run")
print(f"Успішних: {passed} | Невдалих: {failed} | Невиконаних (not_run): {not_run}")
print(f"Усього сценаріїв: {len(CASE_RESULTS)}")

| Критерій | Сценарій | Джерело відповіді | Очікування | Фактичний HTTP | Зміни записів | Стан | Причина |
|---|---|---|---|---|---|---|---|
| AC2 | Список і початковий стан | не застосовується | HTTP 200; items — масив; count == len(items); порядок за id | 200 | 0 (без змін) | passed | усі очікування виконано |
| AC1 | Ручне створення | не застосовується | HTTP 201; поля відповіді == введені; новий id є в списку; записів +1 | 201 | 0 -> 1 (+1) | passed | усі очікування виконано |
| AC3 | Хибний ручний вхід | не застосовується | HTTP 422; status=rejected; сховище не змінене за id і полями | 422 | 1 -> 1 (без змін) | passed | усі очікування виконано |
| AC4 | Коректна AI-пропозиція | підготовлена відповідь | HTTP 201; name/distance_km/difficulty/duration_hours як очікувалися; записів +1 | 201 | 1 -> 2 (+1) | passed | усі очікування виконано |
| AC5 | Неповний або неоднозначний текст | підготовлена відповідь | HTTP 422; status=needs_clarification; missing=['duration_hours']; запису немає

## 11. Самоперевірка і висновок

Допишіть відповіді своїми словами — це частина роботи, а не формальність.

1. **Що довела відповідь сервера, а що довело порівняння зі специфікацією?**
   Код `422` доводить лише те, що сервер відхилив запит. Правильність відмови
   доводить порівняння `status` і `reason` зі специфікацією, а незмінність
   сховища доводить порівняння стану до і після.

2. **Чому незмінної кількості записів недостатньо для перевірки відмови?**
   Кількість записів не зміниться, якщо сервер перезаписав наявний запис
   іншим значенням. Тому знімок містить ідентифікатори й поля.

3. **Який сценарій виявив проблему і що змінилося після виправлення?**
   (заповніть за фактом; якщо проблем не було — поясніть, що саме перевіряли)

4. **Які результати отримані з підготовленими відповідями, а які — з реальною
   моделлю?**
   Підготовлені відповіді доводять **поведінку сервісу** за заданої
   пропозиції, а не якість моделі. Живий виклик доводить зв'язок із
   провайдером, але не стабільність на інших входах.

5. **Чи є `failed` або `not_run`, і який доказ через це відсутній?**

In [11]:
# Відповіді заповнено за фактичними результатами цього проходу.
# Внесок AI-агента у підготовку цих текстів зазначено у README (розділ «Звіт»):
# формулювання варто перечитати й переписати своїми словами перед захистом.
SELF_REVIEW = {
    "1_server_response_vs_specification": (
        "Код відповіді доводить лише те, що сервер прийняв або відхилив запит. "
        "Правильність відмови доводить порівняння змістового поля status і reason зі "
        "специфікацією: 'rejected' з причиною invalid_semantics і 'needs_clarification' "
        "зі списком missing — це різні відмови, і саме вони кажуть, ЧОМУ запис не створено. "
        "А те, що запис справді не створився, доводить порівняння стану сховища до і після."
    ),
    "2_why_count_is_not_enough": (
        "Кількість записів не зміниться, якщо сервер не додав новий, а перезаписав "
        "наявний іншим значенням. Тому знімок стану містить ідентифікатор і всі поля "
        "кожного запису, і відмова перевіряється порівнянням знімків, а не лічильником."
    ),
    "3_problem_found_and_fix": (
        "Під час розробки тести виявили дві проблеми. Перша: FastAPI за замовчуванням "
        "ігнорує зайві поля, тому клієнт міг передати id, а сервер мовчки створив би запис "
        "з іншим ідентифікатором — виправлено extra='forbid'. Друга: форма тіла помилки "
        "розходилася між ручним вводом (список Pydantic) і відмовою AI-функції (вкладений "
        "об'єкт) — виправлено єдиною формою {status, reason, detail}; зміну внесено до "
        "специфікації окремим комітом ДО коду сервісу. Обидві правки мають власні тести."
    ),
    "4_prepared_vs_live": (
        "Підготовлені відповіді (режим stub) доводять поведінку СЕРВІСУ за заданої "
        "пропозиції: перевірку, відхилення, відсутність запису. Вони не доводять нічого "
        "про якість моделі. Живий виклик (режим live) доводить зв'язок сервісу з "
        "провайдером на одному конкретному тексті, але не стабільність на інших входах."
    ),
    "5_failed_or_not_run": (
        "У цьому проході всі детерміновані сценарії пройшли, not_run немає. "
        "Якщо провайдер недоступний, сценарій живої інтеграції буде not_run — це "
        "ВІДСУТНІЙ ДОКАЗ, а не успіх, і в такому разі стверджувати зв'язок із "
        "провайдером не можна."
    ),
}

for key, value in SELF_REVIEW.items():
    print(f"\n### {key}\n{value}")


### 1_server_response_vs_specification
Код відповіді доводить лише те, що сервер прийняв або відхилив запит. Правильність відмови доводить порівняння змістового поля status і reason зі специфікацією: 'rejected' з причиною invalid_semantics і 'needs_clarification' зі списком missing — це різні відмови, і саме вони кажуть, ЧОМУ запис не створено. А те, що запис справді не створився, доводить порівняння стану сховища до і після.

### 2_why_count_is_not_enough
Кількість записів не зміниться, якщо сервер не додав новий, а перезаписав наявний іншим значенням. Тому знімок стану містить ідентифікатор і всі поля кожного запису, і відмова перевіряється порівнянням знімків, а не лічильником.

### 3_problem_found_and_fix
Під час розробки тести виявили дві проблеми. Перша: FastAPI за замовчуванням ігнорує зайві поля, тому клієнт міг передати id, а сервер мовчки створив би запис з іншим ідентифікатором — виправлено extra='forbid'. Друга: форма тіла помилки розходилася між ручним вводом (список Pyda